In [ ]:
import os
import numpy as np
import time
import tensorflow as tf
import keras
from tensorflow.keras.layers import Input, Dense, Dropout, Layer, Conv2D, MaxPooling2D, Bidirectional, LSTM, GRU, TimeDistributed, Flatten
from tensorflow.keras.layers import  MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import tensorflow_model_optimization as tfmot
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

 Copy of data_loading_preprocessing.ipynb
Run the script first to load the samples before executing other functions or scripts

pip install tensorflow_model_optimization

In [ ]:
# Teacher model
def assemble_model(input_shape, n_class):
    model_input = Input(shape=input_shape, dtype='float32', name='name_model_input')

    # Feature extraction part
    x = TimeDistributed(Conv2D(32, kernel_size=(5, 5), activation='relu', data_format='channels_last'))(model_input) 
    x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)
    x = TimeDistributed(Flatten())(x) 
    x = TimeDistributed(Dense(256, activation='gelu'))(x)
    x = Bidirectional(GRU(128, return_sequences=True))(x)
    x = Dropout(0.2)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = Dropout(0.2)(x)

    # Output part
    x = Flatten()(x)
    model_output = Dense(n_class, activation='softmax', name='name_model_output')(x) 

    # Model compiling
    model = Model(inputs=model_input, outputs=model_output)
    # Define the loss function with label smoothing
    loss_function = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.001),
                  loss=loss_function,
                  metrics=['accuracy'])
    return model

# Instantiate the model with the desired number of layers
model_1 = assemble_model(input_shape=(36, 20, 20, 1), n_class=6)

In [ ]:
# Learning Rate Scheduler
def scheduler(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

lr_scheduler = LearningRateScheduler(scheduler)

# Define the EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True)

In [ ]:
# Run the full precision Teacher Model
history1 = model_1.fit(training_generator,
                             batch_size=32,
                             epochs=50,
                             verbose=1, shuffle=True,
                             validation_data=validation_generator, 
                             callbacks=[lr_scheduler, early_stopping])

# Evaluate of Full precision Teacher model
loss, teacher_accuracy = model_1.evaluate(test_generator) # For three test sets

# Save the full precision teacher model
model_1.save("teacher_model.keras")


# Quantization of Teacher Model

Post Training Quantization (PTQ)

In [ ]:
# Create a function that provides input samples for quantized models
def representative_data_gen():
    batch_size = 32
    num_samples = data_train.shape[0]
    
    # Shuffle the data indices
    shuffled_indices = np.arange(num_samples)
    np.random.shuffle(shuffled_indices)
    
    # Iterate over the shuffled data, yielding batches
    for i in range(0, num_samples, batch_size):
        # Get the batch indices
        batch_indices = shuffled_indices[i:i + batch_size]
        # Yield a batch of data
        yield [data_train[batch_indices].astype(np.float32)]

# Post Training Quantization (PTQ)
# Convert the model to TensorFlow Lite format with 8-bit quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model_1)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
converter.inference_input_type = tf.float32 # model is quantized to int8 but can accept input, output float32 datatype 
converter.inference_output_type = tf.float32 
converter.representative_dataset = representative_data_gen

# Disable experimental lowering of tensor list ops
converter._experimental_lower_tensor_list_ops = False

# Generate the TensorFlow Lite model with quantization
tflite_quantized_model = converter.convert()

# Save the quantized model to a file
with open('PTQ_teacher_model.tflite', 'wb') as f:
    f.write(tflite_quantized_model)

print("PTQ teacher model saved successfully.")

Quantization Aware Training (QAT)

In [ ]:
# Prepare Teacher model using  QAT
LastValueQuantizer = tfmot.quantization.keras.quantizers.LastValueQuantizer
MovingAverageQuantizer = tfmot.quantization.keras.quantizers.MovingAverageQuantizer


# TimeDistributed Wrapper
class TimeDistributedQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
    # No weights to quantize for TimeDistributed wrapper
    def get_weights_and_quantizers(self, layer):
        return []

    # Quantize the activations of the underlying layer if necessary
    def get_activations_and_quantizers(self, layer):
        # Assuming the underlying layer has an activation attribute
        if hasattr(layer.layer, 'activation'):
            return [(layer.layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]
        return []

    # No weights to set for TimeDistributed wrapper
    def set_quantize_weights(self, layer, quantize_weights):
        pass

    # Set the quantized activations back to the underlying layer
    def set_quantize_activations(self, layer, quantize_activations):
        if hasattr(layer.layer, 'activation'):
            layer.layer.activation = quantize_activations[0]

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {}
    
# Default Custom Config
class DefaultQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
  
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]

  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]

  def set_quantize_weights(self, layer, quantize_weights):
    layer.kernel = quantize_weights[0]

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}
  

# Convulational layer
class DefaultConv2DQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]

  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]

  def set_quantize_weights(self, layer, quantize_weights):
    layer.kernel = quantize_weights[0]

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]

  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}

class Conv2DQuantizeConfig(DefaultConv2DQuantizeConfig):
  #Change number of bits as needed
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=5, symmetric=True, narrow_range=False, per_axis=False))]
  #Change number of bits as needed. Here same as default
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]



# Dense Layer
class DefaultDenseQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]

  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]

  def set_quantize_weights(self, layer, quantize_weights):
    layer.kernel = quantize_weights[0]

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}    


class DenseQuantizeConfig(DefaultQuantizeConfig):
  #Change number of bits as needed
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=5, symmetric=True, narrow_range=False, per_axis=False))]
  #Change number of bits as needed. Here same as default
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]
  


# Bidirectional layers
class BidirectionalQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
    def __init__(self, forward_quantize_config, backward_quantize_config):
        self.forward_quantize_config = forward_quantize_config
        self.backward_quantize_config = backward_quantize_config

    def get_weights_and_quantizers(self, layer):
        forward_layer = layer.forward_layer
        backward_layer = layer.backward_layer
        return (self.forward_quantize_config.get_weights_and_quantizers(forward_layer) +
                self.backward_quantize_config.get_weights_and_quantizers(backward_layer))

    def get_activations_and_quantizers(self, layer):
        forward_layer = layer.forward_layer
        backward_layer = layer.backward_layer
        return (self.forward_quantize_config.get_activations_and_quantizers(forward_layer) +
                self.backward_quantize_config.get_activations_and_quantizers(backward_layer))

    def set_quantize_weights(self, layer, quantize_weights):
        forward_layer = layer.forward_layer
        backward_layer = layer.backward_layer
        num_forward_weights = len(self.forward_quantize_config.get_weights_and_quantizers(forward_layer))
        self.forward_quantize_config.set_quantize_weights(forward_layer, quantize_weights[:num_forward_weights])
        self.backward_quantize_config.set_quantize_weights(backward_layer, quantize_weights[num_forward_weights:])

    def set_quantize_activations(self, layer, quantize_activations):
        forward_layer = layer.forward_layer
        backward_layer = layer.backward_layer
        num_forward_activations = len(self.forward_quantize_config.get_activations_and_quantizers(forward_layer))
        self.forward_quantize_config.set_quantize_activations(forward_layer, quantize_activations[:num_forward_activations])
        self.backward_quantize_config.set_quantize_activations(backward_layer, quantize_activations[num_forward_activations:])

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {
            'forward_quantize_config': self.forward_quantize_config.get_config(),
            'backward_quantize_config': self.backward_quantize_config.get_config()
        }


# GRU layers    
class GRUQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
    def get_weights_and_quantizers(self, layer):
        return [
            (layer.cell.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False)),
            (layer.cell.recurrent_kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False)),
            (layer.cell.bias, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))
        ]
    def get_activations_and_quantizers(self, layer):
        return []
    def set_quantize_weights(self, layer, quantize_weights):
        layer.cell.kernel = quantize_weights[0]
        layer.cell.recurrent_kernel = quantize_weights[1]
        layer.cell.bias = quantize_weights[2]
    def set_quantize_activations(self, layer, quantize_activations):
        pass
    def get_output_quantizers(self, layer):
        return []
    def get_config(self):
        return {}


# LSTM layers
class DefaultLSTMQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
  def get_weights_and_quantizers(self, layer):
    return [(layer.cell.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False)),
            (layer.cell.recurrent_kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]
  def get_activations_and_quantizers(self, layer):
    return []
  def set_quantize_weights(self, layer, quantize_weights):
    layer.cell.kernel = quantize_weights[0]
    layer.cell.recurrent_kernel = quantize_weights[1]
  def set_quantize_activations(self, layer, quantize_activations):
    pass
  def get_output_quantizers(self, layer):
    return []
  def get_config(self):
    return {}
  
class LSTMQuantizeConfig(DefaultLSTMQuantizeConfig):
  #Change number of bits as needed
  def get_weights_and_quantizers(self, layer):
    return [(layer.cell.kernel, LastValueQuantizer(num_bits=5, symmetric=True, narrow_range=False, per_axis=False)),
            (layer.cell.recurrent_kernel, LastValueQuantizer(num_bits=5, symmetric=True, narrow_range=False, per_axis=False))]
  #Change number of bits as needed. Here same as default
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]
  

# Activation in Dense layers
class DefaultActivationQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))] 

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}

class ActivationQuantizeConfig(DefaultActivationQuantizeConfig):
  def get_weights_and_quantizers(self, layer):
    return []
  def set_quantize_weights(self, layer, quantize_weights):
    return
  #Change number of bits as needed. Here same as default                                      
  def get_activations_and_quantizers (self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]


In [ ]:
# Prepare the QAT-teacher model
quantize_annotate_layer = tfmot.quantization.keras.quantize_annotate_layer
quantize_annotate_model = tfmot.quantization.keras.quantize_annotate_model
quantize_scope = tfmot.quantization.keras.quantize_scope


with quantize_scope(
      {
       'ActivationQuantizeConfig': ActivationQuantizeConfig,
       'DenseQuantizeConfig': DenseQuantizeConfig,
}):
 
       def apply_quantization(layer):
          if isinstance(layer, tf.keras.layers.Dense):
            return tfmot.quantization.keras.quantize_annotate_layer(layer, DenseQuantizeConfig())
          elif isinstance(layer, tf.keras.layers.Activation):
            return tfmot.quantization.keras.quantize_annotate_layer(layer, ActivationQuantizeConfig()) 
          return layer
        
       annotated_model = tf.keras.models.clone_model(
       model_1,clone_function= apply_quantization)
       quant_aware_model_1 = tfmot.quantization.keras.quantize_apply(annotated_model)
       quant_aware_model_1.summary()

In [ ]:
# Complie QAT teacher
quant_aware_model_1.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])
# Run the QAT teacher model
history_QAT= quant_aware_model_1.fit(training_generator,
                             batch_size=32,
                             epochs=100,
                             verbose=1, shuffle=True,
                             validation_data=validation_generator, 
                             callbacks=[lr_scheduler, early_stopping])

# Evaluate the model
loss, teacher_accuracy = quant_aware_model_1.evaluate(test_generator)

In [ ]:
# Convert QAT teacher model  to TensorFlow Lite format with 8-bit quantization
converter_te = tf.lite.TFLiteConverter.from_keras_model(quant_aware_model_1)
converter_te.optimizations = [tf.lite.Optimize.DEFAULT]
converter_te.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
converter_te.inference_input_type = tf.float32 
converter_te.inference_output_type = tf.float32 
converter_te.representative_dataset = representative_data_gen

# Disable experimental lowering of tensor list ops
converter_te._experimental_lower_tensor_list_ops = False

# Generate the TensorFlow Lite model with quantization
tflite_quantized_model_te = converter_te.convert()

# Save the quantized model to a file
with open('QAT_teacher.tflite', 'wb') as f:
    f.write(tflite_quantized_model_te)

print("QAT teacher model saved successfully.")

# Transformer model Generation

In [ ]:
# Full Precision Transformer model
# MultiHeadAttention Layer
# Create positional encoding
def positional_encoding(seq_len, d_model):
    angles = 1 / np.power(10000., (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    positional_encoding = np.zeros((seq_len, d_model))
    positional_encoding[:, 0::2] = np.sin(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 0::2]))
    positional_encoding[:, 1::2] = np.cos(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 1::2]))
    positional_encoding = positional_encoding[np.newaxis, ...]
    return tf.cast(positional_encoding, dtype=tf.float32)

#Create MultiHeadAttention Encoder
class ScaledDotProductAttention(Layer):
    def __init__(self, dropout_ratio=0.1, **kwargs):
        super(ScaledDotProductAttention, self).__init__(**kwargs)
        self.dropout = Dropout(dropout_ratio)

    def call(self, q, k, v, mask=None, training=False):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.dropout(attention_weights, training=training)
        output = tf.matmul(attention_weights, v)
        return output
    
class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads, dropout_ratio=0.1, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0  # d_model must be divisible by num_heads

        self.depth = d_model // self.num_heads
        self.dropout_ratio = dropout_ratio

        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)

        self.dropout = Dropout(dropout_ratio)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v, mask=None, training=False):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)  # (batch_size, seq_len, d_model)
        k = self.wk(k)  # (batch_size, seq_len, d_model)
        v = self.wv(v)  # (batch_size, seq_len, d_model)

        q = self.split_heads(q, batch_size)  # (batch_size, num_heads, seq_len_q, depth)
        k = self.split_heads(k, batch_size)  # (batch_size, num_heads, seq_len_k, depth)
        v = self.split_heads(v, batch_size)  # (batch_size, num_heads, seq_len_v, depth)

        scaled_attention = ScaledDotProductAttention(self.dropout_ratio)(q, k, v, mask, training)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  # (batch_size, seq_len_q, num_heads, depth)
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))  # (batch_size, seq_len_q, d_model)
        output = self.dense(concat_attention)  # (batch_size, seq_len_q, d_model)
        return output
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "dropout_ratio": self.dropout_ratio
        })
        return config


class TransformerMultiHeadAttentionEncoderLayer(Layer):
    def __init__(self, ff_dim, dropout_ratio, **kwargs):
        super(TransformerMultiHeadAttentionEncoderLayer, self).__init__(**kwargs)
        self.ff_dim = ff_dim
        self.dropout_ratio = dropout_ratio

    def build(self, input_shape):
        self.attention = MultiHeadAttention(d_model=input_shape[-1], num_heads=4, dropout_ratio=self.dropout_ratio)
        self.layer_norm1 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(self.dropout_ratio)
        self.dense1 = Dense(self.ff_dim, activation='gelu', kernel_regularizer=l2(0.01))
        self.dense2 = Dense(input_shape[-1], kernel_regularizer=l2(0.01))
        self.layer_norm2 = LayerNormalization(epsilon=1e-6)
        self.dropout2 = Dropout(self.dropout_ratio)
        super(TransformerMultiHeadAttentionEncoderLayer, self).build(input_shape)

    def call(self, inputs, training=False):
        seq_len, d_model = inputs.shape[1], inputs.shape[2]
        pos_encoding = positional_encoding(seq_len, d_model)
        x = inputs + pos_encoding

        attn_output = self.attention(x, x, x)  # Q, K, V are the same for self-attention
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(inputs + attn_output)

        ffn_output = self.dense1(out1)
        ffn_output = self.dense2(ffn_output)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layer_norm2(out1 + ffn_output)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "ff_dim": self.ff_dim,
            "dropout_ratio": self.dropout_ratio
        })
        return config

# Design of Transformer model    
def transformer_model(input_shape, n_class, num_layers=2):
    model_input = Input(shape=input_shape, dtype='float32', name='name_model_input')
    
    # Feature extraction part
    x = TimeDistributed(Flatten())(model_input)
    x = TimeDistributed(Dense(128, activation='gelu'))(x)
    x = TimeDistributed(Dropout(0.5))(x)
    
    # Transformer encoder stack
    for _ in range(num_layers):
        x = TransformerMultiHeadAttentionEncoderLayer(ff_dim=256, dropout_ratio=0.1)(x)
    
    # Output part
    x = Flatten()(x)
    model_output = Dense(n_class, activation='softmax', name='name_model_output')(x)
    
    # Model compiling
    model = Model(inputs=model_input, outputs=model_output)
    # Define the loss function with label smoothing
    loss_function = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.0005),
                  loss = loss_function,
                  metrics=['accuracy'])
    return model

# Instantiate the model with the desired number of layers
model_2 = transformer_model(input_shape=(36, 20, 20, 1), n_class=6, num_layers=2)

In [ ]:
# Run full precision transformer model
history2 = model_2.fit(training_generator,
                             batch_size=32,
                             epochs=50,
                             verbose=1, shuffle=True,
                             validation_data=validation_generator, 
                             callbacks=[lr_scheduler, early_stopping])

# Evaluate the transformer model
loss, student_accuracy = model_2.evaluate(test_generator)

# Save the transformer model
model_2.save("student_model.keras") # for comparing sizes
# Can't save model in keras as some layers built in customized way
model_2.save_weights("student_model_weights.h5")

# Quantization of Student Model

PTQ of Transformer model

In [ ]:
# Post Training Quantization
# Convert the model to TensorFlow Lite format with 8-bit quantization
converter_2 = tf.lite.TFLiteConverter.from_keras_model(model_2)
converter_2.optimizations = [tf.lite.Optimize.DEFAULT]
converter_2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
converter_2.inference_input_type = tf.float32 # model is quantized to int8 but can accept input, output float32 datatype 
converter_2.inference_output_type = tf.float32 
converter_2.representative_dataset = representative_data_gen

# Disable experimental lowering of tensor list ops
converter_2._experimental_lower_tensor_list_ops = False

# Generate the TensorFlow Lite model with quantization
tflite_PTQ_student_model = converter_2.convert()

# Save the quantized model to a file
with open('PTQ_student_model.tflite', 'wb') as f:
    f.write(tflite_PTQ_student_model)

print("PTQ student model saved successfully.")

QAT of transformer model

In [ ]:
# TimeDistributed Wrapper
class TimeDistributedQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
    # No weights to quantize for TimeDistributed wrapper
    def get_weights_and_quantizers(self, layer):
        return []

    # Quantize the activations of the underlying layer if necessary
    def get_activations_and_quantizers(self, layer):
        # Assuming the underlying layer has an activation attribute
        if hasattr(layer.layer, 'activation'):
            return [(layer.layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]
        return []

    # No weights to set for TimeDistributed wrapper
    def set_quantize_weights(self, layer, quantize_weights):
        pass

    # Set the quantized activations back to the underlying layer
    def set_quantize_activations(self, layer, quantize_activations):
        if hasattr(layer.layer, 'activation'):
            layer.layer.activation = quantize_activations[0]

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {}

# Transformer Attention Encoder Layer
class TransformerMultiHeadAttentionEncoderQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
   
  def get_weights_and_quantizers(self, layer):
    # Quantize the weights of the dense and layer normalization layers
    return [
        (layer.dense1.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False)),
        (layer.dense2.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False)),
    ]

  def get_activations_and_quantizers(self, layer):
    # Quantize the activations from the dense and layer normalization layers
    return [
        (layer.dense1.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False)),
        (layer.dense2.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False)),
    ]

  def set_quantize_weights(self, layer, quantize_weights):
    # Assign the quantized weights to the dense and layer normalization layers
    layer.dense1.kernel = quantize_weights[0]
    layer.dense2.kernel = quantize_weights[1]

  def set_quantize_activations(self, layer, quantize_activations):
    # Assign the quantized activations to the dense layers
    layer.dense1.activation = quantize_activations[0]
    layer.dense2.activation = quantize_activations[1]
    
  def get_output_quantizers(self, layer):
    # Define if you want to quantize the output of the layer
    return []

  def get_config(self):
    return {}


# Default Custom Config
class DefaultQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
  
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]

  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]

  def set_quantize_weights(self, layer, quantize_weights):
    layer.kernel = quantize_weights[0]

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}



# Dense Layer
class DefaultDenseQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
   
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]

  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]

  def set_quantize_weights(self, layer, quantize_weights):
    layer.kernel = quantize_weights[0]

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}    

class DenseQuantizeConfig(DefaultQuantizeConfig):

  #Change number of bits as needed
  def get_weights_and_quantizers(self, layer):
    return [(layer.kernel, LastValueQuantizer(num_bits=5, symmetric=True, narrow_range=False, per_axis=False))]
  #Change number of bits as needed. Here same as default
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))]
  

# Activation in Dense layers
class DefaultActivationQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):
    
  def get_activations_and_quantizers(self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=False, narrow_range=False, per_axis=False))] 

  def set_quantize_activations(self, layer, quantize_activations):
    layer.activation = quantize_activations[0]
    
  def get_output_quantizers(self, layer):
    return []

  def get_config(self):
    return {}

class ActivationQuantizeConfig(DefaultActivationQuantizeConfig):

  def get_weights_and_quantizers(self, layer):
    return []
  def set_quantize_weights(self, layer, quantize_weights):
    return
  #Change number of bits as needed. Here same as default                                      
  def get_activations_and_quantizers (self, layer):
    return [(layer.activation, MovingAverageQuantizer(num_bits=8, symmetric=True, narrow_range=False, per_axis=False))]


In [ ]:
quantize_student_layer = tfmot.quantization.keras.quantize_student_layer
quantize_student_model = tfmot.quantization.keras.quantize_student_model
quantize_student_scope = tfmot.quantization.keras.quantize_student_scope

with quantize_student_scope(
      {'ScaledDotProductAttention': ScaledDotProductAttention,
       'MultiHeadAttention': MultiHeadAttention,
       'TimeDistributedQuantizeConfig':TimeDistributedQuantizeConfig,
       'TransformerMultiHeadAttentionEncoderLayer': TransformerMultiHeadAttentionEncoderLayer,
       'TransformerMultiHeadAttentionEncoderQuantizeConfig': TransformerMultiHeadAttentionEncoderQuantizeConfig,
       'ActivationQuantizeConfig': ActivationQuantizeConfig,
       'DenseQuantizeConfig': DenseQuantizeConfig,
}):
 
       def apply_quantization(layer):
          if isinstance(layer, tf.keras.layers.TimeDistributed):
             return tfmot.quantization.keras.quantize_student_layer(layer, TimeDistributedQuantizeConfig())
          elif isinstance(layer, tf.keras.layers.Flatten):
             return layer 
          elif isinstance(layer, tf.keras.layers.Dense):
            return tfmot.quantization.keras.quantize_student_layer(layer, DenseQuantizeConfig())
          elif isinstance(layer, tf.keras.layers.Dropout):
             return layer
          elif isinstance(layer, TransformerMultiHeadAttentionEncoderLayer):
             return tfmot.quantization.keras.quantize_student_layer(layer, TransformerMultiHeadAttentionEncoderQuantizeConfig()) 
          elif isinstance(layer, tf.keras.layers.Activation):
            return tfmot.quantization.keras.quantize_student_layer(layer, ActivationQuantizeConfig()) 
          elif isinstance(layer, tf.keras.layers.Dropout):
             return layer 
          return layer
        
       annotated_student_model = tf.keras.models.clone_model(
       model_2,clone_function= apply_quantization)
       quant_aware_model_2 = tfmot.quantization.keras.quantize_apply(annotated_student_model)
       quant_aware_model_2.summary()

In [ ]:
# Complie the QAT trandformer model
quant_aware_model_2.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

# Run the QAT student model
history_stu_QAT= quant_aware_model_2.fit(training_generator,
                             batch_size=32,
                             epochs=100,
                             verbose=1, shuffle=True,
                             validation_data=validation_generator, 
                             callbacks=[lr_scheduler, early_stopping])

# Evaluate the QAT student model
loss, teacher_accuracy = quant_aware_model_2.evaluate(test_generator)

In [ ]:
# Quantization Aware Training
# Convert the model_1 to TensorFlow Lite format with 8-bit quantization
converter_st = tf.lite.TFLiteConverter.from_keras_model(quant_aware_model_2)
converter_st.optimizations = [tf.lite.Optimize.DEFAULT]
converter_st.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
converter_st.inference_input_type = tf.float32 # model is quantized to int8 but can accept input, output float32 datatype 
converter_st.inference_output_type = tf.float32 
converter_st.representative_dataset = representative_data_gen

# Disable experimental lowering of tensor list ops
converter_st._experimental_lower_tensor_list_ops = False

# Generate the TensorFlow Lite model with quantization
tflite_QAT_student_model = converter_st.convert()

# Save the quantized model to a file
with open('QAT_student.tflite', 'wb') as f:
    f.write(tflite_QAT_student_model)

print("Quantized QAT student model saved successfully.")

Check the Quantized/Full Precision models' results

In [ ]:
# Evaluate Full precision models
def evaluate_model_metrics(model, generator):
    y_true = []

    for i in range(len(generator)):
        _, batch_y = generator[i]
        y_true.append(batch_y)

    y_true = np.concatenate(y_true)
    y_true = np.argmax(y_true, axis=1)
    total_samples = len(y_true)
    start_time = time.time()
    y_pred_probs = model.predict(generator, verbose=1)
    end_time = time.time()
    total_inference_time = end_time - start_time
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Classification metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    # Inference metrics
    avg_latency_ms = (total_inference_time / total_samples) * 1000
    throughput = total_samples / total_inference_time
    # Model complexity
    params = model.count_params()

    # FLOPs
    try:
        from keras_flops import get_flops
        flops = get_flops(model, batch_size=1)
    except:
        flops = "Install keras-flops"

    # Model size
    size_mb = sum([np.prod(w.shape) * w.dtype.itemsize for w in model.get_weights()]) / (1024*1024)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Total Inference Time (s)": total_inference_time,
        "Avg Inference Time per Sample (ms)": avg_latency_ms,
        "Throughput (samples/sec)": throughput,
        "Parameters": params,
        "FLOPs": flops,
        "Model Size (MB)": size_mb
    }

    for k,v in results.items():
        print(f"{k}: {v}")

    return results

In [ ]:
# Evalutate Quantized models
def evaluate_tflite_metrics(interpreter, data_generator, batch_size=32):
    data = data_generator.data
    labels = data_generator.labels
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    y_true = np.argmax(labels, axis=1)
    y_pred_probs = []
    batch_inference_times = []
    total_samples = data.shape[0]
    input_shape = input_details[0]['shape']
    
    print(f"Running inference on {total_samples} samples...")
    
    # Iterate through batches
    for i in range(0, total_samples, batch_size):
        test_batch = data[i:i+batch_size].astype(np.float32)
        current_batch_size = test_batch.shape[0]
        
        # Resize input tensor for current batch
        if current_batch_size != input_shape[0]:
            interpreter.resize_tensor_input(
                input_details[0]['index'], 
                [current_batch_size] + list(input_shape[1:]))
            interpreter.allocate_tensors()
        
        # Time the inference
        start_time = time.time()
        interpreter.set_tensor(input_details[0]['index'], test_batch)
        interpreter.invoke()
        output_batch = interpreter.get_tensor(output_details[0]['index'])
        end_time = time.time()
        
        # Store results
        batch_inference_times.append(end_time - start_time)
        y_pred_probs.append(output_batch)
    
    # Concatenate all predictions
    y_pred_probs = np.concatenate(y_pred_probs, axis=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # Calculate timing metrics
    total_inference_time = sum(batch_inference_times)
    avg_latency_ms = (total_inference_time / total_samples) * 1000
    throughput = total_samples / total_inference_time
    
    # Classification metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    try:
        size_mb = None 
    except:
        size_mb = None
    
    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Total Inference Time (s)": total_inference_time,
        "Avg Inference Time per Sample (ms)": avg_latency_ms,
        "Throughput (samples/sec)": throughput,
        "Model Size (MB)": size_mb,
        "Quantization": "INT8" if input_details[0]['dtype'] == np.int8 else "FLOAT32"
    }
    
    for k, v in results.items():
        print(f"{k}: {v}")
    
    return results, y_pred, y_pred_probs

In [ ]:
# Compare original and quantized model sizes
def get_model_sizes(keras_model_path, tflite_model_path):
    import os
    
    # Original Keras model size
    keras_size_bytes = os.path.getsize(keras_model_path)
    keras_size_mb = keras_size_bytes / (1024 * 1024)
    
    # Quantized TFLite model size
    tflite_size_bytes = os.path.getsize(tflite_model_path)
    tflite_size_mb = tflite_size_bytes / (1024 * 1024)
    
    # Compression ratio
    compression_ratio = keras_size_bytes / tflite_size_bytes
    size_reduction_percent = ((keras_size_bytes - tflite_size_bytes) / keras_size_bytes) * 100
    
    print(f"Original Keras Model: {keras_size_mb:.2f} MB")
    print(f"Quantized TFLite Model: {tflite_size_mb:.2f} MB")
    print(f"Compression Ratio: {compression_ratio:.2f}x")
    print(f"Size Reduction: {size_reduction_percent:.2f}%")
    
    return {
        "keras_size_mb": keras_size_mb,
        "tflite_size_mb": tflite_size_mb,
        "compression_ratio": compression_ratio,
        "size_reduction_percent": size_reduction_percent
    }


In [ ]:
# Check Inference time
def quick_tflite_inference_time(interpreter, data_generator, num_samples=5):
    import time
    import numpy as np
    
    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Get data from generator
    data = data_generator.data
    
    # Resize for single sample inference
    input_shape = input_details[0]['shape']
    interpreter.resize_tensor_input(input_details[0]['index'], [1] + list(input_shape[1:]))
    interpreter.allocate_tensors()
    
    # Warmup (first prediction is slower)
    sample = data[0:1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    
    # Measure
    times = []
    samples_to_test = min(num_samples, data.shape[0])
    
    for i in range(samples_to_test):
        sample = data[i:i+1].astype(np.float32)
        
        start = time.perf_counter()
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        times.append(time.perf_counter() - start)
    
    avg = np.mean(times)
    return avg


In [ ]:
# Load the Interpreters
# Quantized teacher models
# Load the PTQ teacher model
interpreter_1 = tf.lite.Interpreter(model_path='PTQ_teacher_model.tflite')

# Get input and output details
input_details_1 = interpreter_1.get_input_details()
output_details_1 = interpreter_1.get_output_details()

# Resize the input tensor to accept the batch size
batch_size = 32  
input_shape = input_details_1[0]['shape']
input_shape[0] = batch_size
interpreter_1.resize_tensor_input(input_details_1[0]['index'], input_shape)

# Allocate tensors with the new input shape
interpreter_1.allocate_tensors()


# Load the QAT teacher model
interpreter_2 = tf.lite.Interpreter(model_path='QAT_teacher.tflite')
input_details_2 = interpreter_2.get_input_details()
output_details_2 = interpreter_2.get_output_details()
input_shape = input_details_2[0]['shape']
input_shape[0] = batch_size
interpreter_2.resize_tensor_input(input_details_2[0]['index'], input_shape)
interpreter_2.allocate_tensors()


# Quantized student models
# Load the PTQ student model
interpreter_3 = tf.lite.Interpreter(model_path='PTQ_student_model.tflite')
input_details_3  = interpreter_3.get_input_details()
output_details_3  = interpreter_3.get_output_details()
input_shape = input_details_3 [0]['shape']
input_shape[0] = batch_size
interpreter_3.resize_tensor_input(input_details_3 [0]['index'], input_shape)
interpreter_3.allocate_tensors()


# Load the QAT model
interpreter_4 = tf.lite.Interpreter(model_path='QAT_student.tflite')
input_details_4  = interpreter_4.get_input_details()
output_details_4  = interpreter_4.get_output_details()
input_shape = input_details_4 [0]['shape']
input_shape[0] = batch_size
interpreter_4.resize_tensor_input(input_details_4[0]['index'], input_shape)
interpreter_4.allocate_tensors()

In [ ]:
# Full Precision models
teacher_model_test = evaluate_model_metrics(model_1, test_generator) 
student_model_test = evaluate_model_metrics(model_2, test_generator)

In [ ]:
# Quantized models
PTQ_teacher_test = evaluate_tflite_metrics(interpreter_1, test_generator) 
QAT_teacher_test = evaluate_tflite_metrics(interpreter_2, test_generator) 
PTQ_student_test = evaluate_tflite_metrics(interpreter_3, test_generator) 
QAT_student_test = evaluate_tflite_metrics(interpreter_4, test_generator) 

In [ ]:
# Size compare of full precision and quantized models
teacher_size_PTQ = get_model_sizes("teacher_model.keras", "PTQ_teacher_model.tflite")
teacher_size_QAT = get_model_sizes("teacher_model.keras", "QAT_teacher_model.tflite")
student_size_PTQ = get_model_sizes("student_model.keras", "PTQ_student_model.tflite")
student_size_QAT = get_model_sizes("student_model.keras", "QAT_student_model.tflite")